In [7]:
import pandas as pd
from eurostat_client import fetch_dataset, COUNTRIES, COUNTRY_NAMES

In [18]:
test_df = fetch_dataset('nrg_ind_ef ')
test_df.head(5)

[eurostat_client] Fetching nrg_ind_ef ...


AttributeError: 'NoneType' object has no attribute 'get'

In [12]:
test_df = fetch_dataset('nrg_pc_205')
test_df.head(5)

[eurostat_client] Fetching nrg_pc_205...
[eurostat_client] nrg_pc_205: 2895 rows, 83 columns


,freq,siec,nrg_cons,unit,tax,currency,geo\TIME_PERIOD,2007-S1_value,2007-S1_flag,2007-S2_value,...,2023-S2_value,2023-S2_flag,2024-S1_value,2024-S1_flag,2024-S2_value,2024-S2_flag,2025-S1_value,2025-S1_flag,2025-S2_value,2025-S2_flag
0,S,E7000,MWH20-499,KWH,I_TAX,EUR,AL,NaN,:,NaN,...,0.1468,e,0.1503,,0.1551,e,0.1703,,NaN,:
1,S,E7000,MWH20-499,KWH,I_TAX,EUR,AT,NaN,:,0.1412,...,0.3163,e,0.2791,e,0.2693,e,0.2785,e,0.2784,e
2,S,E7000,MWH20-499,KWH,I_TAX,EUR,BA,NaN,:,NaN,...,0.1147,,0.1173,,0.1207,,0.1448,,0.1435,
3,S,E7000,MWH20-499,KWH,I_TAX,EUR,BE,NaN,:,0.1435,...,0.2875,,0.2507,,0.2560,,0.2800,,0.2611,
4,S,E7000,MWH20-499,KWH,I_TAX,EUR,BG,NaN,:,0.0767,...,0.1759,,0.1500,,0.1699,,0.1667,,0.1809,


In [11]:
test_df[['freq', 'siec', 'nrg_cons', 'unit', 'tax', 'currency',
       'geo\TIME_PERIOD', '2007-S1_value', '2007-S1_flag', '2007-S2_value',
       '2007-S2_flag', '2008-S1_value',]].head(3)

,freq,siec,nrg_cons,unit,tax,currency,geo\TIME_PERIOD,2007-S1_value,2007-S1_flag,2007-S2_value,2007-S2_flag,2008-S1_value
0,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,AL,NaN,:,NaN,:,NaN
1,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,AT,NaN,:,0.2008,,0.2005
2,S,E7000,KWH1000-2499,KWH,I_TAX,EUR,BA,NaN,:,NaN,:,NaN


In [1]:
import pandas as pd
from eurostat_client import fetch_dataset, COUNTRIES, COUNTRY_NAMES
from transformer import melt_eurostat_wide

PIPELINE = 'electricity_prices'

# nrg_pc_204 = households, nrg_pc_205 = industry
DATASETS = {
    'household': 'nrg_pc_204',
    'industry':  'nrg_pc_205',
}


total_rows = 0
try:
    frames = []
    for consumer_type, code in DATASETS.items():
        df_raw = fetch_dataset(code)

        # Typical dimension columns in this dataset:
        # 'freq', 'unit', 'tax', 'currency', 'nrg_cons', 'geo\TIME_PERIOD'
        # Rename geo column — Eurostat uses backslash in column name
        geo_col = [c for c in df_raw.columns if 'geo' in c.lower() or 'TIME' in c][0]
        df_raw = df_raw.rename(columns={geo_col: 'geo'})

        id_cols = ['freq', 'unit', 'tax', 'currency', 'nrg_cons', 'geo']
        id_cols = [c for c in id_cols if c in df_raw.columns]

        df = melt_eurostat_wide(df_raw, id_cols=id_cols, value_name='price_eur_kwh')

        # Filter to our countries and annual data
        df = df[df['geo'].isin(COUNTRIES)]
        if 'freq' in df.columns:
            df = df[df['freq'] == 'S']  # S = semi-annual (price data is per semester)

        df['consumer_type'] = consumer_type
        df['country_code']  = df['geo']
        df['country_name']  = df['geo'].map(COUNTRY_NAMES)
        df['tax_included']  = df.get('tax', pd.Series(dtype=str)).apply(
            lambda x: True if str(x).upper() in ('I_TAX', 'TAX') else False
        )
        df['price_band'] = df.get('nrg_cons', pd.Series(dtype=str))

        keep = ['country_code', 'country_name', 'year', 'semester',
                'consumer_type', 'price_band', 'price_eur_kwh', 'tax_included', 'flag']
        df = df[[c for c in keep if c in df.columns]]

        frames.append(df)

    final = pd.concat(frames, ignore_index=True)

except Exception as e:
    print (e)
    raise

[eurostat_client] Fetching nrg_pc_204...
[eurostat_client] nrg_pc_204: 2196 rows, 83 columns
[eurostat_client] Fetching nrg_pc_205...
[eurostat_client] nrg_pc_205: 2895 rows, 83 columns


In [2]:
final

,country_code,country_name,year,semester,consumer_type,price_band,price_eur_kwh,tax_included,flag
0,CZ,Czechia,2007,S1,household,KWH1000-2499,0.1573,True,NaN
1,DE,Germany,2007,S1,household,KWH1000-2499,0.2258,True,NaN
2,EU27_2020,European Union (27),2007,S1,household,KWH1000-2499,0.1974,True,NaN
3,HR,Croatia,2007,S1,household,KWH1000-2499,0.1223,True,NaN
4,LU,Luxembourg,2007,S1,household,KWH1000-2499,0.2219,True,NaN
...,...,...,...,...,...,...,...,...,...
114049,PL,Poland,2025,S2,industry,TOT_KWH,0.2806,False,NaN
114050,PT,Portugal,2025,S2,industry,TOT_KWH,0.1705,False,NaN
114051,RO,Romania,2025,S2,industry,TOT_KWH,0.2940,False,NaN
114052,SE,Sweden,2025,S2,industry,TOT_KWH,0.0741,False,NaN
